# Canon BIN to NIfTI Converter

Convert Canon raw linearized .bin files (CHI and Fund) to NIfTI format for QuantUS analysis.

## Features
- Extracts both CHI (CEUS) and Fund (B-mode) data from Canon .bin files
- Creates **dual-file output** for optimal workflow:
  - **Raw files** (`*_RAW.nii`): Original float64 values for accurate quantitative analysis
  - **Normalized files** (`*_normalized_GUI.nii`): uint8 (0-255) for GUI visualization
- Handles Canon's extreme dynamic range with percentile-based normalization
- Preserves proper orientation for QuantUS

## Workflow
1. Edit file paths in the Configuration section
2. Run all cells to generate both RAW and normalized NIfTI files
3. **For ROI drawing**: Load `*_normalized_GUI.nii` files in main QuantUS GUI
4. **For analysis**: Load the same `*_normalized_GUI.nii` files in QuantUS-Plugins-CEUS
   - The analysis pipeline automatically detects and uses the corresponding `*_RAW.nii` files for calculations
   - This ensures accurate quantification while maintaining proper visualization

## Why Two Files?
- **Visualization**: Main GUI needs normalized uint8 data (0-255) to display properly
- **Analysis**: Quantification metrics need raw float64 values for accuracy
- **Automatic**: The analysis loader detects `*_RAW.nii` files automatically - you only interact with normalized files!

## 1. Setup and Imports

In [1]:
import struct
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import os
from pathlib import Path

## 2. Configuration

**Edit these paths to match your files:**

In [2]:
# === EDIT THESE PATHS ===
chi_file = "/Users/samantha/Downloads/OneDrive_2_11-11-2025/R20240116132619009RawLinear_CHI.bin"
fund_file = "/Users/samantha/Downloads/OneDrive_1_11-11-2025 2/R20240116132629903RawLinear_Fund.bin"
output_dir = "/Users/samantha/Desktop/ultrasound lab stuff/bin_test"

# Verify files exist
print(f"CHI file exists: {os.path.exists(chi_file)}")
print(f"Fund file exists: {os.path.exists(fund_file)}")

CHI file exists: True
Fund file exists: True


## 3. Extraction Function

In [3]:
def extract_canon_bin(bin_path, output_nifti_path=None, normalize_chi=True, verbose=True):
    """
    Extract Canon .bin file to NIfTI format.
    
    Args:
        bin_path: Path to Canon .bin file
        output_nifti_path: Optional path for NIfTI output
        normalize_chi: If True, normalize CHI (CEUS) data to 0-255 using percentile-based scaling
        verbose: Print extraction progress
    
    Returns:
        volume_data: 3D numpy array (axial, lateral, frames)
        metadata: Dictionary with header information
    """
    if verbose:
        print(f"Reading: {Path(bin_path).name}")
    
    # Read header (20 bytes: 5 integers)
    with open(bin_path, 'rb') as f:
        hdr_info = struct.unpack('i', f.read(4))[0]
        axial_size = struct.unpack('i', f.read(4))[0]
        lateral_size = struct.unpack('i', f.read(4))[0]
        frame_num = struct.unpack('i', f.read(4))[0]
        data_type = struct.unpack('i', f.read(4))[0]
    
    # Determine data type: 1=CHI (float64), 2=Fund (uint16)
    data_type_str = "CHI (CEUS)" if data_type == 1 else "Fund (B-mode)"
    read_dtype = np.float64 if data_type == 1 else np.uint16
    
    if verbose:
        print(f"  Dimensions: {axial_size} x {lateral_size} x {frame_num} frames")
        print(f"  Type: {data_type_str}")
    
    # Read frame data
    num_per_frame = axial_size * lateral_size
    volume_data = np.zeros((axial_size, lateral_size, frame_num), dtype=np.float64)
    
    frames_read = 0
    with open(bin_path, 'rb') as f:
        f.seek(20)  # Skip header
        for i in range(frame_num):
            frame = np.fromfile(f, dtype=read_dtype, count=num_per_frame)
            if frame.size != num_per_frame:
                break
            
            # Convert to float64 if needed
            if data_type == 2:
                frame = frame.astype(np.float64)
            
            # Reshape to (axial, lateral)
            frame = frame.reshape((axial_size, lateral_size))
            volume_data[:, :, i] = frame
            frames_read = i + 1
    
    if verbose:
        print(f"  Raw value range: {volume_data.min():.6f} to {volume_data.max():.6f}")
    
    # Apply standard rotation (GUI will handle final orientation)
    volume_data = np.rot90(volume_data, k=3, axes=(0, 1))
    volume_data = volume_data.transpose(1, 0, 2)
    
    # Normalize CHI data using percentile-based scaling to handle extreme dynamic range
    if data_type == 1 and normalize_chi:
        p_low = np.percentile(volume_data, 20)
        p_high = np.percentile(volume_data, 95)
        
        if verbose:
            print(f"  Normalizing CHI: p20={p_low:.6f}, p95={p_high:.6f}")
        
        volume_data = np.clip(volume_data, p_low, p_high)
        volume_data = ((volume_data - p_low) / (p_high - p_low) * 255)
        
        if verbose:
            print(f"  After normalization: mean={volume_data.mean():.2f}, median={np.median(volume_data):.2f}")
    
    if verbose:
        print(f"  Extracted {frames_read} frames")
        print(f"  Final shape: {volume_data.shape}")
    
    # Save to NIfTI if path provided
    if output_nifti_path:
        affine = np.eye(4)
        nii = nib.Nifti1Image(volume_data, affine)
        nii.header['pixdim'] = [1.0, 0.0993, 0.0725, 0.1307, 1.0, 0.0, 0.0, 0.0]
        nib.save(nii, output_nifti_path)
        if verbose:
            print(f"  Saved: {output_nifti_path}")
    
    metadata = {
        'shape': volume_data.shape,
        'data_type_str': data_type_str,
        'frames_read': frames_read
    }
    
    return volume_data, metadata

## 4. Extract and Convert Files

Run these cells to convert your Canon .bin files to NIfTI format:

In [4]:
# Extract CHI (CEUS) - RAW values, no normalization
chi_output = os.path.join(output_dir, "CHI_extracted_RAW.nii")
chi_data, chi_meta = extract_canon_bin(chi_file, chi_output, normalize_chi=False)

Reading: R20240116132619009RawLinear_CHI.bin
  Dimensions: 604 x 524 x 514 frames
  Type: CHI (CEUS)
  Raw value range: 0.000000 to 0.664440
  Extracted 514 frames
  Final shape: (604, 524, 514)
  Saved: /Users/samantha/Desktop/ultrasound lab stuff/bin_test/CHI_extracted_RAW.nii


In [5]:
# Extract Fund (B-mode)
fund_output = os.path.join(output_dir, "Fund_extracted.nii")
fund_data, fund_meta = extract_canon_bin(fund_file, fund_output, normalize_chi=False)

Reading: R20240116132629903RawLinear_Fund.bin
  Dimensions: 604 x 524 x 514 frames
  Type: Fund (B-mode)
  Raw value range: 0.000000 to 3481.000000
  Extracted 514 frames
  Final shape: (604, 524, 514)
  Saved: /Users/samantha/Desktop/ultrasound lab stuff/bin_test/Fund_extracted.nii


## 5. Create Normalized Versions for GUI Display

The normalized versions allow the main QuantUS GUI to display the images properly. The analysis pipeline in QuantUS-Plugins-CEUS will automatically use the raw files for calculations when you load the normalized files.

In [ ]:
# Create normalized versions for GUI display
print("Creating normalized versions for GUI...")

# Normalize CHI using percentile clipping (handles extreme dynamic range)
chi_p20 = np.percentile(chi_data, 20)
chi_p95 = np.percentile(chi_data, 95)
chi_normalized = np.clip(chi_data, chi_p20, chi_p95)
chi_normalized = ((chi_normalized - chi_p20) / (chi_p95 - chi_p20) * 255)
print(f"  CHI: p20={chi_p20:.6f}, p95={chi_p95:.6f} -> 0-255")

# Normalize Fund using min-max scaling
fund_normalized = ((fund_data - fund_data.min()) / (fund_data.max() - fund_data.min()) * 255)
print(f"  Fund: {fund_data.min():.2f}-{fund_data.max():.2f} -> 0-255")

# Save normalized versions
chi_norm_output = os.path.join(output_dir, "CHI_normalized_GUI.nii")
fund_norm_output = os.path.join(output_dir, "Fund_normalized_GUI.nii")

nib.save(nib.Nifti1Image(chi_normalized, np.eye(4)), chi_norm_output)
nib.save(nib.Nifti1Image(fund_normalized, np.eye(4)), fund_norm_output)

print(f"\n✓ Saved normalized versions:")
print(f"  {chi_norm_output}")
print(f"  {fund_norm_output}")
print(f"\n📊 Use these for GUI visualization")
print(f"📈 Use *_RAW.nii files for quantitative analysis")